In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.8 The Solar System: N-Body Gravitation and the Long Game

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.8",
    title="The Solar System: N-Body Gravitation and the Long Game",
    blurb="Nine bodies, one force law, and questions only a computer can "
    "answer: whether the clockwork stays clockwork over a thousand years, "
    "which integrators can be trusted with a millennium, and how much of "
    "Mercury's famous perihelion drift plain Newtonian gravity explains "
    "before relativity claims the rest.",
    difficulty="intermediate",
    estimate="120–150 min",
)

## Notebook overview

[§1.4](kepler-orbits.ipynb) solved the two-body problem and closed it with a
theorem: bound orbits are ellipses, fixed in space, forever. The real solar
system violates that theorem everywhere, gently: every planet pulls on every
other, ellipses precess, and the question of whether the whole arrangement
is stable resisted the greatest analysts of three centuries before becoming,
in the 1980s, a question for computers {cite}`laskar1989`. This notebook
builds the full N-body problem: all eight planets and the Sun, one force
law, no approximations beyond the discretization of time.

Two threads from earlier notebooks come due. First, the integrator lesson of
[§1.6](integrators.ipynb): over a thousand years the difference between a
symplectic method and a "better" adaptive one is not accuracy but
*character*, bounded energy wobble versus secular drift, and we measure both.
Second, the Mercury thread: [§2.4](../02-classical-mechanics/central-force.ipynb)
explains *why* a perturbed orbit precesses and Volume IV's capstone computes
relativity's famous 43 arcseconds per century; here we compute the much
larger *Newtonian* share, the ~530 arcseconds per century that Mercury's
perihelion drifts because Venus, Jupiter, and the rest exist, and assemble
the historical ledger those numbers settle. The craft on display is
measurement discipline: our integrator precesses orbits *numerically*, so we
measure that bias on a problem whose answer is exactly known (the two-body
system: zero) and subtract it. The standard reference for everything deeper
is Murray & Dermott {cite}`murraydermott`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the answer
is wrong; it means the output did not match what the check expected, which
may be a genuine error, a different-but-valid convention, or too tight a
tolerance. Treat a ✗ as a prompt to locate the discrepancy. Passing is
strong evidence, not proof.

## Theory in brief

**The N-body problem.** Bodies $i = 0, \dots, N-1$ with masses $m_i$ at
positions $\mathbf r_i$ obey

```{math}
:label: eq-ss-nbody
\ddot{\mathbf r}_i \;=\; G\sum_{j \neq i} m_j\,
\frac{\mathbf r_j - \mathbf r_i}{|\mathbf r_j - \mathbf r_i|^3} ,
```

pairwise inverse-square attraction and nothing else. Total energy, momentum,
and angular momentum are conserved exactly; almost nothing else is. For
$N \ge 3$ there is no general closed-form solution, which is why this
notebook exists.

**Astronomical units.** Measuring length in AU, time in years, and mass in
solar masses makes the numbers tame and fixes $G$: for a test mass on a
circular orbit of radius $1$ AU and period $1$ yr, Kepler's third law
$T^2 = 4\pi^2 a^3 / G M_\odot$ forces

```{math}
:label: eq-ss-units
G \;=\; 4\pi^2 \;\approx\; 39.478
\quad \text{AU}^3\,M_\odot^{-1}\,\text{yr}^{-2} .
```

**Building orbits from elements.** An orbit of semi-major axis $a$ and
eccentricity $e$ has perihelion distance $r_p = a(1 - e)$, and the vis-viva
equation $v^2 = G(M + m)(2/r - 1/a)$ gives the perihelion speed

```{math}
:label: eq-ss-visviva
v_p \;=\; \sqrt{\frac{G (M_\odot + m)}{a}\,\frac{1 + e}{1 - e}} ,
```

perpendicular to the radius. Launching each planet at its perihelion with
{eq}`eq-ss-visviva` builds a *model* solar system, coplanar, with each
planet's real $(a, e)$ but arbitrary perihelion directions: the right
masses, sizes, and shapes, without pretending to be an ephemeris. For the
secular (orbit-averaged) questions this notebook asks, that is the physics
that matters.

**Symplectic integration and the long game.** [§1.6](integrators.ipynb)
showed velocity Verlet's energy error *oscillating* while Euler's grew;
the deep reason is that Verlet is symplectic (it exactly conserves a
slightly-wrong Hamiltonian, so the energy of the true one can only wobble).
Adaptive Runge–Kutta is more accurate per step and *not* symplectic: its
small per-step energy errors accumulate with one sign, a secular drift that
no tolerance setting removes, only postpones. For a thousand-year
integration the distinction is the whole game.

**Precession and the LRL vector.** For pure $1/r$ attraction the
Laplace–Runge–Lenz vector of [§1.4](kepler-orbits.ipynb),

```{math}
:label: eq-ss-lrl
\mathbf A \;=\; \mathbf v \times \mathbf L - \frac{GM_\odot\,
\hat{\mathbf r}}{1} ,
```

(per unit mass, heliocentric) is a constant pointing at perihelion, so its
angle is a precession *meter*: any secular rotation of $\mathbf A$ measures
departure from closed ellipses, whether caused by other planets (real) or
by the integrator (artifact). The perturbing planets drive Mercury's
perihelion forward at a rate classical secular theory puts near
$532''$/century (Venus $\approx 278$, Jupiter $\approx 154$, Earth
$\approx 90$, the rest small); general relativity adds $43''$
([§4.8](../04-special-relativity/gr-capstone.ipynb)); together they close
the observed budget that once had astronomers hunting for the phantom
planet Vulcan.

## Setup

Masses (in $M_\odot$), semi-major axes (AU), and eccentricities for the
eight planets are the standard fact-sheet values. Everything integrates in
the barycentric frame (total momentum removed); heliocentric quantities are
differences from the Sun's state. The N-body acceleration is one
vectorized NumPy expression; the integrator is the velocity Verlet of
[§1.6](integrators.ipynb), and no randomness appears anywhere in this
notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import animate, draw, validate

G_AU = 4.0 * np.pi**2  # eq-ss-units: G in AU^3 / (M_sun yr^2)

# (name, mass [M_sun], a [AU], e) — NASA planetary fact sheet
PLANETS = [
    ("Mercury", 1.6601e-7, 0.3871, 0.2056),
    ("Venus", 2.4478e-6, 0.7233, 0.0068),
    ("Earth", 3.0035e-6, 1.0000, 0.0167),
    ("Mars", 3.2272e-7, 1.5237, 0.0934),
    ("Jupiter", 9.5479e-4, 5.2026, 0.0485),
    ("Saturn", 2.8586e-4, 9.5549, 0.0556),
    ("Uranus", 4.3662e-5, 19.218, 0.0472),
    ("Neptune", 5.1514e-5, 30.110, 0.0087),
]


def build_system(names=None):
    """Assemble Sun + planets, each launched at perihelion by vis-viva.

    Each selected planet starts at its perihelion distance a(1−e) with the
    exact perihelion speed of eq-ss-visviva, perpendicular to its radius;
    perihelion directions are spread by the golden angle so no artificial
    alignment biases the perturbations. The whole system is shifted to the
    barycentric frame (total momentum and centre of mass removed), so the
    Sun wobbles as it truly does.

    Parameters
    ----------
    names : list of str or None
        Planet names to include (None means all eight).

    Returns
    -------
    tuple of numpy.ndarray
        ``(m, r, v)``: masses (n,), positions (n, 2) in AU, velocities
        (n, 2) in AU/yr; index 0 is the Sun.
    """
    chosen = [p for p in PLANETS if names is None or p[0] in names]
    n = len(chosen) + 1
    m = np.zeros(n)
    r = np.zeros((n, 2))
    v = np.zeros((n, 2))
    m[0] = 1.0
    golden = np.pi * (3.0 - np.sqrt(5.0))
    for i, (_, mass, a, e) in enumerate(chosen, start=1):
        m[i] = mass
        r_p = a * (1.0 - e)
        v_p = np.sqrt(G_AU * (1.0 + mass) * (1.0 + e) / (a * (1.0 - e)))
        th = golden * i
        c, s = np.cos(th), np.sin(th)
        r[i] = r_p * np.array([c, s])
        v[i] = v_p * np.array([-s, c])
    v -= (m[:, None] * v).sum(axis=0) / m.sum()
    r -= (m[:, None] * r).sum(axis=0) / m.sum()
    return m, r, v


def nbody_accel(m, r):
    """All N accelerations of eq-ss-nbody in one broadcast expression.

    Builds the (n, n, 2) array of separations r_j − r_i, the (n, n) inverse
    cube distances with the diagonal zeroed, and contracts against the
    masses: O(N²) pairwise gravity with no Python loop over pairs.

    Parameters
    ----------
    m : numpy.ndarray
        Masses, shape (n,), in M_sun.
    r : numpy.ndarray
        Positions, shape (n, 2), in AU.

    Returns
    -------
    numpy.ndarray
        Accelerations, shape (n, 2), in AU/yr².
    """
    d = r[None, :, :] - r[:, None, :]
    dist2 = (d**2).sum(axis=-1)
    np.fill_diagonal(dist2, 1.0)
    inv3 = dist2**-1.5
    np.fill_diagonal(inv3, 0.0)
    return G_AU * (d * (m[None, :, None] * inv3[:, :, None])).sum(axis=1)


def verlet_orbits(m, r0, v0, dt, n_steps, sample_every=1):
    """Integrate eq-ss-nbody with velocity Verlet, sampling states.

    The symplectic kick–drift–kick scheme of §1.6: exactly time-reversible,
    with bounded (oscillating, non-secular) energy error — the property
    that makes millennium-scale orbit integration honest.

    Parameters
    ----------
    m : numpy.ndarray
        Masses, shape (n,).
    r0, v0 : numpy.ndarray
        Initial positions and velocities, shape (n, 2); copied, not mutated.
    dt : float
        Time step in years.
    n_steps : int
        Number of steps.
    sample_every : int, optional
        Keep every k-th state (default 1).

    Returns
    -------
    tuple of numpy.ndarray
        ``(t, R, V)``: sample times (s,), positions (s, n, 2), velocities
        (s, n, 2).
    """
    r = r0.copy()
    v = v0.copy()
    a = nbody_accel(m, r)
    ts, rs, vs = [0.0], [r.copy()], [v.copy()]
    for k in range(1, n_steps + 1):
        v += 0.5 * dt * a
        r += dt * v
        a = nbody_accel(m, r)
        v += 0.5 * dt * a
        if k % sample_every == 0:
            ts.append(k * dt)
            rs.append(r.copy())
            vs.append(v.copy())
    return np.array(ts), np.array(rs), np.array(vs)


def total_energy(m, r, v):
    """Total mechanical energy of the system, kinetic plus pairwise potential.

    The conserved quantity of eq-ss-nbody, and this notebook's principal
    instrument: its drift (or bounded wobble) diagnoses the integrator.

    Parameters
    ----------
    m : numpy.ndarray
        Masses, shape (n,).
    r, v : numpy.ndarray
        Positions and velocities, shape (n, 2).

    Returns
    -------
    float
        Energy in M_sun AU²/yr².
    """
    kin = 0.5 * float((m * (v**2).sum(axis=1)).sum())
    d = r[None, :, :] - r[:, None, :]
    dist = np.sqrt((d**2).sum(axis=-1))
    iu = np.triu_indices(len(m), k=1)
    pot = -G_AU * float((m[iu[0]] * m[iu[1]] / dist[iu]).sum())
    return kin + pot


def lrl_angle_series(r_helio, v_helio):
    """Unwrapped angle of the heliocentric LRL vector along a trajectory.

    Evaluates eq-ss-lrl per unit mass at each sample and returns the
    unwrapped polar angle of A: constant for pure two-body motion, drifting
    linearly under secular perturbation (or integrator bias) — the
    precession meter of §1.4, put to work.

    Parameters
    ----------
    r_helio, v_helio : numpy.ndarray
        Heliocentric position and velocity samples, shape (s, 2).

    Returns
    -------
    numpy.ndarray
        Unwrapped LRL angles in radians, shape (s,).
    """
    L = r_helio[:, 0] * v_helio[:, 1] - r_helio[:, 1] * v_helio[:, 0]
    rad = np.hypot(r_helio[:, 0], r_helio[:, 1])
    ax_ = v_helio[:, 1] * L - G_AU * r_helio[:, 0] / rad
    ay_ = -v_helio[:, 0] * L - G_AU * r_helio[:, 1] / rad
    return np.unwrap(np.arctan2(ay_, ax_))


RAD_PER_YR_TO_AS_PER_CY = np.degrees(1.0) * 3600.0 * 100.0

## Exercise 1 — One force law, vectorized and certified

Everything downstream rests on `nbody_accel` computing {eq}`eq-ss-nbody`
correctly, so we certify it against the physics of
[§1.4](kepler-orbits.ipynb) before trusting it with nine bodies.

**Part a)** Verify the unit system: {eq}`eq-ss-units` gives $G = 4\pi^2 =
39.4784\ldots$ exactly by construction; confirm that a test mass
($m = 10^{-12}\,M_\odot$) on the circular orbit $r = (1, 0)$ AU,
$v = (0, 2\pi)$ AU/yr around a solar mass at the origin feels acceleration
of magnitude exactly $G$ from `nbody_accel` (`rtol=1e-12`), pointing in
$-\hat{\mathbf x}$.

**Part b)** Integrate that two-body system for one year with `verlet_orbits`
at $\Delta t = 10^{-4}$ yr and verify the test mass returns to its starting
point to within $|\Delta \mathbf r| < 10^{-5}$ AU: the definition of the
year, recovered by the machinery that will now be pointed at the real
system.

**Part c)** Verify the conservation laws on a two-planet system (Sun,
Earth, Jupiter; $\Delta t = 5 \times 10^{-4}$ yr, $20$ yr): total energy from
`total_energy` conserved to $|\Delta E/E| < 10^{-8}$, total momentum
$\sum m\mathbf v$ zero to $10^{-12}$ (it started zero and Verlet's
pairwise forces cannot create any), and total angular momentum
$\sum m (x v_y - y v_x)$ conserved to relative $10^{-10}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    a_mag,
    G_AU,
    "on the unit circular orbit the acceleration magnitude is exactly "
    "G = 4π², the unit system's defining identity",
    rtol=1e-12,
)
validate.check(
    a1[1, 0] < 0.0 and abs(a1[1, 1]) < 1e-15,
    "and it points at the Sun: pure −x̂ at (1, 0)",
    f"a = ({a1[1, 0]:.4f}, {a1[1, 1]:.1e})",
)
validate.check(
    closure < 1e-5,
    "the circular orbit closes after exactly one year: the year, recovered",
    f"|Δr| = {closure:.2e} AU",
)
validate.check(
    dE_rel < 1e-8 and float(np.hypot(*P_c)) < 1e-12 and dL_rel < 1e-10,
    "energy, momentum, and angular momentum conserved on Sun+Earth+Jupiter "
    "over 20 years at the stated tolerances",
    f"|ΔE/E| {dE_rel:.1e}, |P| {float(np.hypot(*P_c)):.1e}, " f"|ΔL/L| {dL_rel:.1e}",
)

## Exercise 2 — Nine bodies, five years

Now the whole system. `build_system()` launches all eight planets at their
perihelia by {eq}`eq-ss-visviva` ({numref}`fig-ss-launch` shows the
construction for one orbit), spreads the perihelion directions by the
golden angle, and hands the Sun its barycentric recoil.

**Part a)** Integrate the full nine-body system with `verlet_orbits` for
$5$ yr at $\Delta t = 10^{-4}$ yr (sampling every 100 steps) and verify
total energy is
conserved to $|\Delta E/E| < 10^{-8}$ and the barycentre stays put
($|\sum m \mathbf r|/M_{\rm tot} < 10^{-10}$ AU): with every planet
pulling on every other, the conservation laws are the only guarantee not
negotiated away.

**Part b)** Plot the orbits (inner system and full system panels), and
animate the inner four planets with fading trails: five years is four
Mercury years plus a bit, and genuine motion is exactly what a still
cannot show. The animation's physics is validated through Part a)'s
conserved quantities, which are computed from the same trajectory the
player draws.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    dE_full < 1e-8,
    "nine bodies, five years: total energy conserved to the stated 1e-8",
    f"|ΔE/E| = {dE_full:.2e}",
)
validate.check(
    com_max < 1e-10,
    "the barycentre does not move: momentum started at zero and pairwise "
    "forces cannot manufacture any",
    f"max |R_com| = {com_max:.2e} AU",
)

## Exercise 3 — Kepler's third law, from the machine

Kepler read $T^2 \propto a^3$ off Tycho's tables; we read it off our own
integration. The clean way to measure a period from a partial arc is the
*mean motion*: the heliocentric polar angle $\theta(t)$ of a planet grows
by $2\pi$ per orbit, so the slope of the unwrapped angle
(`numpy.unwrap` of `numpy.arctan2`, then a linear `numpy.polyfit`) is
$n = 2\pi/T$ even when the arc is a fraction of an orbit. (For an
eccentric orbit $\dot\theta$ oscillates within each revolution, so the fit
wants whole orbits to average over; the integration below is long enough
that even Neptune completes two.)

**Part a)** Integrate the full system for $350$ yr at $\Delta t =
2\times10^{-3}$ yr, sampling every 25 steps — the sampling interval of
$0.05$ yr keeps Mercury's angle advancing by well under $\pi$ per sample
even at perihelion, which `numpy.unwrap` requires (coarser sampling
aliases Mercury's $0.24$-yr orbit and silently wrecks the fit). For each
planet, measure $T_i = 2\pi/n_i$ from the fitted mean motion of its
heliocentric angle.

**Part b)** Verify each measured period against Kepler's third law in the
form $T = a^{3/2}/\sqrt{1 + m}$ yr (the two-body result with the planet's
own mass correction) to `rtol=1e-2`: Jupiter and Saturn tug everyone, so
exact agreement is neither expected nor found, and the sub-percent
residual *is* the mutual perturbation. Then fit $\log T$ against $\log a$
(`numpy.polyfit`, degree 1) across all eight planets and verify the slope
is $3/2$ within $\pm 0.01$: the law survives the interactions that blur
its individual instances.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    periods,
    T_kepler,
    "every measured period meets T = a^(3/2)/√(1+m) at the percent level; "
    "the residuals are the mutual perturbations themselves",
    rtol=1e-2,
)
validate.close(
    slope_kepler,
    1.5,
    "the fitted log T–log a slope across all eight planets is Kepler's 3/2",
    atol=0.01,
    rtol=0.0,
)

## Exercise 4 — The long game: symplectic character versus accuracy

[§1.6](integrators.ipynb) promised that the reason "molecular dynamics uses
velocity Verlet" would come due when integrations got long. Here it comes
due. We integrate the Sun–Earth–Jupiter three-body system for $1000$ years
two ways and watch not the *size* of the energy error but its *shape*.

**Part a)** Velocity Verlet at $\Delta t = 2\times10^{-3}$ yr
($5\times10^5$ steps). Record the relative energy error $|E(t) - E_0|/|E_0|$
every 500 steps. Verify it stays below $10^{-7}$ for the entire millennium
*and* shows no trend: the mean error over the last hundred years
(`numpy.mean` over the sampled errors in each window) must be within a
factor of $3$ of the mean over the first hundred (a wobble revisits its
floor; a drift does not).

**Part b)** Adaptive Runge–Kutta (`scipy.integrate.solve_ivp`, RK45,
`rtol=1e-6`, `atol=1e-9`, dense sampling every $2$ yr) on the identical
system. Verify its final energy error *exceeds* Verlet's maximum by a
factor above $30$, and that its error grows monotonically in time (the
error at $250$-yr checkpoints strictly increasing): per step it is the
better integrator; per millennium it is the wrong one. The moral, plotted
in {numref}`fig-ss-longgame`, is the deepest lesson this notebook has to
teach, and it travels far beyond gravity: the molecular-dynamics
simulations of Volume V's horizon live and die by it.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    float(err_v.max()) < 1e-7 and late < 3.0 * early,
    "the symplectic millennium: Verlet's energy error stays below 1e-7 and "
    "shows no secular trend (late mean within 3× the early mean)",
    f"max {float(err_v.max()):.1e}, early {early:.1e}, late {late:.1e}",
)
validate.check(
    float(err_rk[-1]) > 30.0 * float(err_v.max()),
    "RK45's drift overwhelms Verlet's wobble by more than 30× at year 1000",
    f"ratio {float(err_rk[-1]) / float(err_v.max()):.0f}×",
)
validate.check(
    all(a < b for a, b in zip(checkpoints, checkpoints[1:])),
    "and the drift is secular: strictly increasing at every 250-year "
    "checkpoint — one-signed accumulation, not a wobble",
    f"checkpoints {[f'{c:.1e}' for c in checkpoints]}",
)

## Exercise 5 — A thought experiment: Jupiter at a thousand times the mass

How fragile is the arrangement? The time-honoured numerical experiment
(run in every computational-physics course since the machines could manage
it) is brutal and illuminating: multiply Jupiter's mass by $1000$ — making
it a $0.955\,M_\odot$ companion star at $5.2$ AU, and the Sun half of a
binary — and watch what survives. The answer is *not* "nothing": a planet
deep inside a binary can orbit one star stably, and the boundary between
safe and doomed has been mapped numerically, most famously by Holman and
Wiegert {cite}`holmanwiegert1999`, whose fitted critical semi-major axis
for orbits around one star of a coplanar binary (mass fraction $\mu$,
eccentricity $e_b$, separation $a_b$) is

```{math}
:label: eq-ss-critical
a_c = a_b \left(0.464 - 0.380\,\mu - 0.631\,e_b + 0.586\,\mu e_b
      + 0.150\,e_b^2 - 0.198\,\mu e_b^2\right),
```

their least-squares fit to a grid of long numerical integrations — an
equation *born* from the kind of experiment this exercise runs. For our
heavy Jupiter, $\mu = 0.488$ and $e_b = 0.049$ give $a_c \approx 1.36$ AU:
Earth at $1.00$ AU should survive, and Mars at $1.52$ AU should not.

**Part a)** Integrate Sun, Earth, Mars, and the normal Jupiter with
`verlet_orbits` for $50$ yr at $\Delta t = 10^{-3}$ yr and record both
planets' heliocentric distances.
Verify Earth stays within the narrow annulus $[0.97, 1.04]$ AU and Mars
within $[1.35, 1.70]$ AU (each planet's eccentricity-driven excursion plus
Jupiter's real, gentle nudging).

**Part b)** Rebuild the same system with $m_{\rm Jup} \to 1000\,m_{\rm
Jup}$ (the barycentric launch adjusts consistently) and integrate $200$ yr
at $\Delta t = 5\times10^{-4}$ yr. Compute $a_c$ from
{eq}`eq-ss-critical` and verify the two fates it predicts: Earth's
distance stays bounded within $[0.85, 1.15]$ AU — jostled, but bound —
while Mars is ejected outright, its maximal heliocentric distance
exceeding $30$ AU (past Neptune's orbit; in fact it leaves by thousands of
AU). The stability boundary of the new binary runs *between* the two
orbits, and the comparison figure tells the story: the ordinary solar
system's tameness is a property of its mass ratios, not of gravity.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    0.97 < float(rE_n.min())
    and float(rE_n.max()) < 1.04
    and 1.35 < float(rM_n.min())
    and float(rM_n.max()) < 1.70,
    "with the real Jupiter, fifty years keep Earth inside [0.97, 1.04] AU "
    "and Mars inside [1.35, 1.70] AU",
    f"Earth [{rE_n.min():.4f}, {rE_n.max():.4f}], "
    f"Mars [{rM_n.min():.4f}, {rM_n.max():.4f}]",
)
validate.check(
    1.0 < a_crit < 1.524,
    "the Holman–Wiegert critical radius of the new binary falls between "
    "the two orbits: a_c separates Earth from Mars",
    f"1.0 < a_c = {a_crit:.3f} < 1.524 AU",
)
validate.check(
    0.85 < float(rE_h.min()) and float(rE_h.max()) < 1.15,
    "Earth, inside a_c, survives two centuries beside a companion star: "
    "jostled but bound in [0.85, 1.15] AU",
    f"[{rE_h.min():.3f}, {rE_h.max():.3f}]",
)
validate.check(
    float(rM_h.max()) > 30.0,
    "Mars, outside a_c, is ejected: it leaves the planetary system past "
    "Neptune's orbit",
    f"r_max = {rM_h.max():.0f} AU",
)

## Exercise 6 — The Galilean clockwork: the Laplace resonance

Jupiter runs a solar system in miniature, and its three inner large
moons keep the most famous appointment in celestial mechanics. Io,
Europa, and Ganymede orbit with sidereal periods $1.769138$,
$3.551181$, and $7.154553$ days — near the ratio $1\!:\!2\!:\!4$, but
*near* is the wrong word: the pairwise ratios are $2.0073$ and
$2.0147$, off from $2$ in the third digit. The true lock, found by
Laplace, is not in the pairs but in the *combination* of mean motions
$n_i = 2\pi/T_i$:

```{math}
:label: eq-ss-laplace
n_{\rm Io} - 3\,n_{\rm Eu} + 2\,n_{\rm Ga} \;=\; 0 ,
```

exact to observational precision (the associated resonance angle
librates about $180°$, so a triple conjunction of the three moons can
*never* occur). The resonance has a thermal consequence worth the
detour: by continually exchanging momentum at the same orbital phases,
it holds Io and Europa on eccentric orbits that Jupiter's tides would
otherwise have circularized long ago. A moon on an eccentric orbit is
kneaded — stretched and relaxed every orbit — and the dissipated work
has to go somewhere: roughly $100$ terawatts in Io, which makes it the
most volcanically active body in the solar system, and enough in
Europa to maintain a liquid-water ocean beneath its ice shell. Peale,
Cassen, and Reynolds published the prediction that tides should melt
Io {cite}`peale1979` three days before Voyager 1 arrived and
photographed the volcanic plumes: one of the cleanest called shots in
planetary science, and it begins with the arithmetic of
{eq}`eq-ss-laplace`.

**Part a)** The arithmetic. From the three quoted sidereal periods,
form the mean motions $n_i = 2\pi/T_i$ and verify: the pairwise ratios
equal $2.00729$ and $2.01470$ (`rtol=1e-4`) — genuinely not $2$ — while
the Laplace combination $|n_{\rm Io} - 3n_{\rm Eu} + 2n_{\rm Ga}|/
n_{\rm Io}$ is below $10^{-5}$: four orders of magnitude smaller than
the pairwise offsets. The resonance lives in the combination.

**Part b)** The measurement. Put the miniature system in the machine:
rebuild `PLANETS` (temporarily, restoring it afterwards as in Exercise
5) with Jupiter as the central mass and the three moons at their real
masses (in $M_{\rm Jup}$: $4.70\times10^{-5}$, $2.53\times10^{-5}$,
$7.81\times10^{-5}$), semi-major axes (in units of $10^6$ km: $0.4217$,
$0.6710$, $1.0704$), and eccentricities ($0.0041$, $0.0090$,
$0.0013$). With the central mass set to one, the unit system of
{eq}`eq-ss-units` works unchanged — $G = 4\pi^2$ simply defines the
time unit. Integrate $60$ Io orbits with `verlet_orbits` at $\Delta t =
2\times10^{-4}$ (sampling every $50$ steps keeps Io's unwrapped angle
well under $\pi$ per sample), measure the three mean motions with the
unwrap-and-fit of Exercise 3, and verify: the measured ratios match the
real ones to `rtol=1e-3`, and the measured Laplace combination stays
at least $30$ times smaller than the smallest pairwise offset
$|n_{\rm Io}/n_{\rm Eu} - 2|$. (Our launch does not phase-lock the
moons — that would require placing them at the librating configuration
— so the measured combination is small rather than zero; the point the
integration makes is that the near-cancellation is carried by the
orbits themselves, not by a numerical coincidence.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([ratio_ie, ratio_eg]),
    np.array([2.00729, 2.01470]),
    "the quoted periods give pairwise ratios 2.00729 and 2.01470: "
    "genuinely not 2, at the third digit",
    rtol=1e-4,
)
validate.check(
    combo_real < 1e-5,
    "yet the Laplace combination n_Io - 3n_Eu + 2n_Ga cancels four "
    "orders more deeply than the pairwise offsets: eq-ss-laplace",
    f"|combo|/n_Io = {combo_real:.1e}",
)
validate.close(
    np.array([ratio_ie_m, ratio_eg_m]),
    np.array([ratio_ie, ratio_eg]),
    "the 60-orbit integration of the miniature system reproduces both "
    "real mean-motion ratios",
    rtol=1e-3,
)
validate.check(
    combo_meas < pair_offset / 30.0,
    "and carries the same deep cancellation: the measured combination "
    "sits far below the pairwise offsets, without any phase-locking "
    "by hand",
    f"combo {combo_meas:.1e} vs pair offset {pair_offset:.1e}",
)

## Exercise 7 — Mercury's perihelion: the Newtonian share, measured

The Mercury thread's penultimate number. Mercury's perihelion drifts
forward at about $575''$ per century relative to the fixed stars; classical
secular theory attributes $\approx 532''$ to the pulls of the other
planets, and the unexplained remainder of $43''$ was a scandal for half a
century before general relativity claimed it exactly
([§4.8](../04-special-relativity/gr-capstone.ipynb) computes it). This
exercise measures the Newtonian share from our own nine-body integration,
using the LRL precession meter {eq}`eq-ss-lrl`.

The measurement demands the discipline this course keeps preaching: the
integrator itself precesses eccentric orbits (a $\Delta t^2$ artifact of
Verlet, large for Mercury's $e = 0.206$), so the raw LRL drift is
meaningless. The honest protocol measures the *instrument's bias* on a
problem with a known answer, then subtracts:

**Part a)** The control. Integrate the Sun–Mercury *two-body* system for
$60$ yr at $\Delta t = 2\times10^{-4}$ yr, extract Mercury's heliocentric
LRL angle series with `lrl_angle_series`, and fit its slope with
`numpy.polyfit` (degree 1). For pure two-body motion the true answer is
*exactly zero* ([§1.4](kepler-orbits.ipynb)'s closed-orbit theorem), so
the fitted slope, several thousand arcseconds per century, is pure
integrator bias; record it. Confirm it is an artifact by verifying it
shrinks by a factor $\approx 4$ (within a factor window $[3, 5]$) when
$\Delta t$ is halved on a shorter $20$-yr control: the $\Delta t^2$
signature.

**Part b)** The measurement. Integrate the full nine-body system with the
*same* $\Delta t = 2\times10^{-4}$ yr for the same $60$ yr, fit Mercury's
LRL slope, and subtract the control. Verify the difference lands in the
window $[480, 640]''$/century, comfortably containing the classical
secular value $\approx 532''$ (our coplanar model with spread perihelia
reproduces it to several percent; the residual is model geometry, not
error). Convert with the module constant `RAD_PER_YR_TO_AS_PER_CY`.

**Part c)** Close the ledger in print: Newtonian share (measured here)
$+43''$ (relativity, [§4.8](../04-special-relativity/gr-capstone.ipynb))
versus the observed $\approx 575''$/century. The sum this notebook cannot
supply is exactly the sum Volume IV does; between them, no Vulcan
required.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    3.0 < dt2_ratio < 5.0,
    "the control drift is a Δt² integrator artifact: halving the step cuts "
    "it by ≈ 4 on the matched 20-yr runs",
    f"ratio {dt2_ratio:.2f}",
)
validate.check(
    480.0 < newtonian_share < 640.0,
    "the control-subtracted Newtonian precession of Mercury lands on "
    "classical secular theory's ≈ 532″/century",
    f"measured {newtonian_share:.0f}″/century",
)
validate.check(
    abs(newtonian_share + 43.0 - 575.0) < 75.0,
    "and the ledger closes: Newtonian share + relativity's 43″ meets the "
    "observed ≈ 575″/century with no Vulcan required",
    f"sum {newtonian_share + 43.0:.0f}″ vs observed ≈ 575″",
)

## Notebook summary

- The unit system {eq}`eq-ss-units` was certified ($|\mathbf a| = 4\pi^2$
  on the unit circle to $10^{-12}$; the circular year closed to
  $10^{-5}$ AU), and the conservation laws held on demand (energy $10^{-8}$,
  momentum $10^{-12}$, angular momentum $10^{-10}$).
- The full nine-body system, launched from real $(a, e)$ by vis-viva
  {eq}`eq-ss-visviva`, ran five years with energy conserved to $10^{-8}$
  and an immobile barycentre; 350 years of it returned every planet's
  period on Kepler's $T = a^{3/2}$ to the percent (the residuals being the
  mutual perturbations) with a fitted log–log slope of $1.4989$.
- The thousand-year Sun–Earth–Jupiter contest settled the
  [§1.6](integrators.ipynb) promise:
  Verlet's energy error wobbled below $10^{-7}$ with no secular trend
  while RK45's drifted monotonically past it by more than $30\times$;
  per-step accuracy and long-run trustworthiness are different virtues.
- Multiplying Jupiter's mass by $1000$ turned the Sun into half of a
  binary star whose Holman–Wiegert stability radius, $a_c \approx 1.36$ AU
  by {eq}`eq-ss-critical`, runs *between* Earth and Mars — and the
  integration delivered exactly the two fates the formula predicts: Earth
  jostled but bound in $[0.85, 1.15]$ AU, Mars ejected past Neptune within
  two centuries. Stability is a property of mass ratios, with a
  quantitative boundary.
- Jupiter's own miniature solar system carried the Laplace resonance:
  the quoted Galilean periods gave pairwise ratios $2.00729$ and
  $2.01470$ (not $2$!) while the combination $n_{\rm Io} - 3n_{\rm Eu}
  + 2n_{\rm Ga}$ cancelled to below $10^{-5}$ of $n_{\rm Io}$, and our
  own 60-orbit integration reproduced both facts; the same resonance
  maintains the eccentricities whose tidal kneading powers Io's
  volcanoes and Europa's ocean.
- Mercury's Newtonian perihelion precession was *measured*, with the
  course's own epistemology: the integrator's bias (thousands of
  arcseconds per century of pure artifact, verified $\Delta t^2$ by
  step-halving) was calibrated on the two-body problem, whose true answer
  is exactly zero, and subtracted, leaving $562''$/century against
  secular theory's $\approx 532''$; with relativity's $43''$
  ([§4.8](../04-special-relativity/gr-capstone.ipynb)) the observed
  $\approx 575''$ budget closes, and Vulcan stays unnecessary.

## Outlook

- **Chaos on gigayear scales.** Laskar's integrations {cite}`laskar1989`
  showed the inner solar system is chaotic with a Lyapunov time near
  $5$ Myr: the clockwork is statistical at long range. The tools for
  *quantifying* that sensitivity (Lyapunov exponents, from a subject this
  course meets next in the nonlinear-dynamics notebook) turn this
  notebook's integrations into measurements of predictability itself.
- **Resonances.** The restricted three-body structure of
  [§2.9](../02-classical-mechanics/lagrange-points.ipynb) organizes the
  asteroid belt: Jupiter's mean-motion resonances carve the Kirkwood gaps,
  an experiment one Kirkwood-gap notebook could run with exactly this
  machinery {cite}`murraydermott`.
- **Better long-game integrators.** Wisdom–Holman splitting integrates the
  Kepler part exactly and only the perturbations numerically, buying
  centuries per step; it is the professional descendant of the symplectic
  lesson here.
- **The molecular long game.** Volume V's molecular-dynamics horizon runs
  on the same verdict: thermodynamic averages need trajectories whose
  energy wobbles rather than drifts, which is why velocity Verlet is the
  standard there too.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()